# A股量化系统 - 快速入门

本 Notebook 演示：
1. 验证数据库连接
2. 通过 Tushare Pro 拉取行情数据
3. 用 Polars 做基本分析
4. 写入 TimescaleDB 并查询

In [1]:
import os
import sys
sys.path.insert(0, '/app')   # from app.xxx 可直接使用

import tushare as ts
import polars as pl
import pandas as pd
from loguru import logger

ts.set_token(os.environ.get('TUSHARE_TOKEN', ''))
pro = ts.pro_api()

print(f'Tushare version : {ts.__version__}')
print(f'Polars  version : {pl.__version__}')

Tushare version : 1.4.25
Polars  version : 1.39.0


In [2]:
# ── 1. 验证数据库连接 ────────────────────────────────────────
from app.utils.db import health_check

ok = health_check()
print('数据库连接:', '✅ 正常' if ok else '❌ 失败')

2026-03-16 20:31:02.409 | INFO     | app.utils.db:make_engine:32 - 数据库引擎已创建: timescaledb:5432/quant_db


数据库连接: ✅ 正常


In [3]:
# ── 2. 拉取平安银行日线数据 ──────────────────────────────────
df_pd = pro.daily(
    ts_code='000001.SZ',
    start_date='20230101',
    end_date='20241231',
)
df = pl.from_pandas(df_pd)
print(df.schema)
df.head()

Schema([('ts_code', String), ('trade_date', String), ('open', Float64), ('high', Float64), ('low', Float64), ('close', Float64), ('pre_close', Float64), ('change', Float64), ('pct_chg', Float64), ('vol', Float64), ('amount', Float64)])


ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""000001.SZ""","""20241231""",11.93,11.99,11.7,11.7,11.95,-0.25,-2.0921,1.4754e6,1.7472e6
"""000001.SZ""","""20241230""",11.78,11.97,11.78,11.95,11.83,0.12,1.0144,1.3518e6,1.6109e6
"""000001.SZ""","""20241227""",11.87,11.9,11.66,11.83,11.86,-0.03,-0.253,1.2900e6,1.5184e6
"""000001.SZ""","""20241226""",11.92,11.93,11.78,11.86,11.92,-0.06,-0.5034,1000074.7,1.1837e6
"""000001.SZ""","""20241225""",11.86,12.02,11.84,11.92,11.86,0.06,0.5059,1.4753e6,1.7600e6


In [4]:
# ── 3. Polars 简单分析：计算 20 日均线 ───────────────────────
df_analysis = (
    df
    .rename({'trade_date': 'date', 'vol': 'volume', 'pct_chg': 'pct_change'})
    .sort('date')
    .with_columns([
        pl.col('close').rolling_mean(20).alias('ma20'),
        pl.col('close').rolling_mean(60).alias('ma60'),
        pl.col('volume').rolling_mean(5).alias('vol_ma5'),
    ])
)
df_analysis.tail(10)

ts_code,date,open,high,low,close,pre_close,change,pct_change,volume,amount,ma20,ma60,vol_ma5
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""000001.SZ""","""20241218""",11.58,11.74,11.57,11.65,11.53,0.12,1.0408,1.0166e6,1.1863e6,11.511,11.5195,990890.974
"""000001.SZ""","""20241219""",11.59,11.64,11.54,11.59,11.65,-0.06,-0.515,697379.04,808465.664,11.511,11.549167,933119.864
"""000001.SZ""","""20241220""",11.59,11.7,11.58,11.62,11.59,0.03,0.2588,714646.27,831437.46,11.528,11.577833,807290.54
"""000001.SZ""","""20241223""",11.64,11.84,11.64,11.73,11.62,0.11,0.9466,1.6594e6,1.9535e6,11.5555,11.606,978027.936
"""000001.SZ""","""20241224""",11.72,11.87,11.72,11.86,11.73,0.13,1.1083,1.3508e6,1.5957e6,11.585,11.630833,1.0878e6
"""000001.SZ""","""20241225""",11.86,12.02,11.84,11.92,11.86,0.06,0.5059,1.4753e6,1.7600e6,11.6115,11.6545,1.1795e6
"""000001.SZ""","""20241226""",11.92,11.93,11.78,11.86,11.92,-0.06,-0.5034,1000074.7,1.1837e6,11.6375,11.666333,1.2400e6
"""000001.SZ""","""20241227""",11.87,11.9,11.66,11.83,11.86,-0.03,-0.253,1.2900e6,1.5184e6,11.66,11.673167,1.3551e6
"""000001.SZ""","""20241230""",11.78,11.97,11.78,11.95,11.83,0.12,1.0144,1.3518e6,1.6109e6,11.688,11.668833,1.2936e6


In [5]:
# ── 4. 拉取并写入 TimescaleDB ────────────────────────────────
from app.data_pipeline.fetch_daily import fetch_stock_daily, upsert_daily

df_stock = fetch_stock_daily('000001.SZ', '20230101', '20241231')
n = upsert_daily(df_stock)
print(f'写入记录数: {n}')

2026-03-16 20:31:02.930 | INFO     | app.data_pipeline.fetch_daily:fetch_stock_daily:35 - 拉取 000001.SZ 日线数据: 20230101 ~ 20241231
2026-03-16 20:31:03.188 | SUCCESS  | app.data_pipeline.fetch_daily:fetch_stock_daily:49 - 获取 484 条记录
2026-03-16 20:31:03.414 | SUCCESS  | app.data_pipeline.fetch_daily:upsert_daily:79 - 写入/更新 484 条日线记录


写入记录数: 484


In [6]:
# ── 5. 从 TimescaleDB 查询，用 Polars 原生接口读取 ───────────
import os

df_query = pl.read_database_uri(
    query="""
        SELECT time, symbol, close, volume, pct_change
        FROM market.daily
        WHERE symbol = '000001.SZ'
        ORDER BY time DESC
        LIMIT 20
    """,
    uri=os.environ['DATABASE_URL'],
)
df_query

time,symbol,close,volume,pct_change
"datetime[μs, UTC]",str,"decimal[38,10]",i64,"decimal[38,10]"
2024-12-31 00:00:00 UTC,"""000001.SZ""",11.7000000000,1475367,-2.0921000000
2024-12-30 00:00:00 UTC,"""000001.SZ""",11.9500000000,1351846,1.0144000000
2024-12-27 00:00:00 UTC,"""000001.SZ""",11.8300000000,1290012,-0.2530000000
2024-12-26 00:00:00 UTC,"""000001.SZ""",11.8600000000,1000075,-0.5034000000
2024-12-25 00:00:00 UTC,"""000001.SZ""",11.9200000000,1475283,0.5059000000
…,…,…,…,…
2024-12-10 00:00:00 UTC,"""000001.SZ""",11.7900000000,2167807,1.0283000000
2024-12-09 00:00:00 UTC,"""000001.SZ""",11.6700000000,964063,0.0858000000
2024-12-06 00:00:00 UTC,"""000001.SZ""",11.6600000000,1726269,1.9231000000
